## Imports

In [ ]:
from pathlib import Path
import json
import pickle
import random
import math

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tokenizers import Tokenizer

### load preprocessing files

In [2]:
PROJECT_DIR = Path.cwd().parent

PROCESSED_DIR = PROJECT_DIR/"data"/"processed"
MODEL_DIR = PROJECT_DIR/"models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

hf_tokenizer = Tokenizer.from_file(str(PROCESSED_DIR/"tokenizer.json"))

with open(PROCESSED_DIR/"encoded_lines.pkl", "rb") as f:
    encoded_lines = pickle.load(f)

with open(PROCESSED_DIR/"vocab.json", "r", encoding="utf-8") as f:
    vocab_data = json.load(f)

word_to_id = vocab_data["word_to_id"]
id_to_word = {int(idx): word for word, idx in word_to_id.items()}

vocab_size = len(word_to_id)

bos_id = word_to_id["<bos>"]
eos_id = word_to_id["<eos>"]
unk_id = word_to_id["<unk>"]
pad_id = word_to_id["<pad>"]

print("Vocabulary size:", vocab_size)

for split, sequences in encoded_lines.items():
    print(split, "sequences:", len(sequences))

Vocabulary size: 46165
test sequences: 2185
train sequences: 17556
validation sequences: 1841


---

## Training

### settings

In [3]:
# reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# model settings
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.2

# memory-conscious sequence settings
BLOCK_SIZE = 32
BATCH_SIZE = 4

# training settings
LEARNING_RATE = 0.001
EPOCHS = 3

print("Block size:", BLOCK_SIZE)
print("Batch size:", BATCH_SIZE)

Using device: cuda
Block size: 32
Batch size: 4


block size means the LSTM processes up to 32 input tokens in each training example.   
and cuz batch size is 4, it processes four examples at once.

### making the dataset ready

In [ ]:
def make_lstm_dataset(sequences, block_size, pad_id):
    input_blocks = []
    target_blocks = []

    for sequence in sequences:
        # input and target are shifted by one token.
        for start in range(0, len(sequence) - 1, block_size):
            
            end = min(start + block_size, len(sequence) - 1) # never letting the input include the very last token of the sequence
            
            # input and target are shifted  by one position
            x = sequence[start:end]
            y = sequence[start + 1: end + 1]
            
            # adding padding, to make the blocks same length
            x = x + [pad_id]*(block_size - len(x))
            y = y + [pad_id]*(block_size - len(y))

            input_blocks.append(x)
            target_blocks.append(y)
            
    # converting to tensor
    inputs = torch.tensor(input_blocks, dtype=torch.long)
    targets = torch.tensor(target_blocks, dtype=torch.long)

    return TensorDataset(inputs, targets)

# for each split
train_dataset = make_lstm_dataset(encoded_lines["train"], BLOCK_SIZE, pad_id)
validation_dataset = make_lstm_dataset(encoded_lines["validation"], BLOCK_SIZE, pad_id)
test_dataset = make_lstm_dataset(encoded_lines["test"], BLOCK_SIZE, pad_id)

print("Training blocks:", len(train_dataset))
print("Validation blocks:", len(validation_dataset))
print("Test blocks:", len(test_dataset))

# we only shuffle training blocks
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# one example
x_example, y_example = train_dataset[0]

print("\nInput ids:", x_example[:10].tolist())
print("Target ids:", y_example[:10].tolist())

Training blocks: 70766
Validation blocks: 7386
Test blocks: 8390

Input ids: [2, 21646, 119, 3840, 152, 44, 1, 4345, 23, 749]
Target ids: [21646, 119, 3840, 152, 44, 1, 4345, 23, 749, 44]


### Defining the lstm

In [9]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1, dropout=0.2, pad_id=0):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id) # first embeddings
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0) # then learning patterns across the sequence
        self.dropout = nn.Dropout(dropout) # to reduce overfitting during training
        
        # score for every possible next token
        self.output = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, input_ids):
        embedded = self.embedding(input_ids) # [batch_size, sequence_length, embedding_dim]
        lstm_output, _ = self.lstm(embedded) # [batch_size, sequence_length, hidden_dim]
        lstm_output = self.dropout(lstm_output) 
        logits = self.output(lstm_output)
        
        return logits

### Initialize the model and loss function

In [10]:
model = LSTMLanguageModel(vocab_size=vocab_size, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT, pad_id=pad_id).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id, reduction="sum") # ignoring padded targets when calculating loss
optimizer = torch.optim.Adam(model.parameters(), lr= LEARNING_RATE)

number_of_parameters = sum(parameter.numel() for parameter in model.parameters())

print(f"Model parameters: {number_of_parameters:,}")

Model parameters: 11,996,501


---

### training and evaluation

In [ ]:
def run_epoch(model, data_loader, criterion, optimizer=None):
    training = optimizer is not None
    
    if training:
        model.train()
    else:
        model.eval()
    
    total_loss = 0.0
    total_tokens = 0
    
    for inputs, targets in data_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        if training:
            optimizer.zero_grad()
            
        with torch.set_grad_enabled(training):
            logits = model(inputs)
            
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1)) # for cross entropy
            
            if training:
                valid_tokens = (targets != pad_id).sum()
                average_loss = loss / valid_tokens.clamp(min=1)
                
                average_loss.backward()
                
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
            total_loss += loss.item()
            total_tokens += (targets != pad_id).sum().item()
        
        average_loss = total_loss / total_tokens
        perplexity = math.exp(min(average_loss, 20))
        
        return {
            "loss": average_loss,
            "perplexity": perplexity
        }